In [1]:
from app.db_connector import get_engine
from app.icog_util import remove_stop_words
from app.models import Entity, SubTopic, SubTopic_Entity_Link
from app.transformers_util import generate_embeddings, get_util, get_model
from sqlalchemy.orm import Session
from sqlalchemy import (
    select,
    or_,
    func,
)

engine = get_engine()
st_util = get_util()
st_model = get_model()

2024-04-09 10:44:03 - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
2024-04-09 10:44:03 - Starting new HTTPS connection (1): huggingface.co:443
2024-04-09 10:44:03 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json HTTP/1.1" 200 0
2024-04-09 10:44:03 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json HTTP/1.1" 200 0
2024-04-09 10:44:04 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/README.md HTTP/1.1" 200 0
2024-04-09 10:44:04 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json HTTP/1.1" 200 0
2024-04-09 10:44:04 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/sentence_bert_config.json HTTP/1.1" 200 0
2024-04-09 10:44:04 - https://huggingface.co:443 "HEAD /sentence-transformers/all-MiniLM-L6-v2/resolve/main/co

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2024-04-09 10:44:23 - Connecting to database using TCP
2024-04-09 10:44:23 - Connected to database using TCP


In [55]:
with Session(engine) as session:
    entities = session.execute(select(Entity.name, Entity.description, func.array_agg(Entity.id)).filter(
        or_(Entity.type == 'concepts', Entity.type == 'topic')).group_by(Entity.name, Entity.description)).all()
    

sentences = []
for entity in entities:
    text = entity.name + ' - ' + entity.description
    summary = remove_stop_words(text)
    sentences.append(text)
            

In [56]:
sentences

['vector database - A vector database is a type of database that stores and retrieves vector data, which can be used for semantic search and other machine learning tasks',
 'LLM - LLM (Language Model) is a machine learning model used for natural language processing tasks',
 'competition - The article suggests that competition is not the pathway to success, but creation is',
 "distractions - Things that divert one's attention away from the task at hand",
 'Vector indexes - Vector indexes are a way of organizing and searching for vectors in a high-dimensional space, based on their similarity to a given query vector.',
 'Socratic method - A teaching method developed by the ancient Greek philosopher Socrates, which involves asking a series of questions to stimulate critical thinking and to illuminate ideas',
 'few-shot prompting - Few-shot prompting is a machine learning technique in which a model is trained on a small number of examples and then used to make predictions on new, unseen dat

In [25]:
sentences_embedding = st_model.encode(sentences, convert_to_tensor=True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [26]:
clusters = st_util.community_detection(sentences_embedding, min_community_size=2, threshold=0.6)

In [34]:
subtopics = []
for ci, cluster in enumerate(clusters):
    print(f"cluster {ci}")
    subtopic = ""
    for i in cluster:
        subtopic += sentences[i] + "\n"  
        print(sentences[i])
    print("\n\n")
    subtopics.append(subtopic)
    

cluster 0
distraction-free virtual meetings - Virtual meetings that are free from distractions and interruptions
distraction-free meetings - Meetings that are free from distractions and interruptions, allowing for better engagement and productivity
distraction-free meetings - Meetings that are free from distractions and allow for full engagement from attendees
virtual meetings - Meetings held through video conferencing technology
virtual meetings - Meetings that are held remotely through video conferencing technology



cluster 1
Vector indexes - Vector indexes are a way of organizing and searching for vectors in a high-dimensional space, based on their similarity to a given query vector.
Vector search retrieval methods - Vector search retrieval methods are a way of searching for information based on the similarity of vectors in a high-dimensional space.
Keyword indexes - Keyword indexes are a way of organizing and searching for text based on the presence of specific keywords or phrase

In [47]:
print(subtopics[1])

Vector indexes - Vector indexes are a way of organizing and searching for vectors in a high-dimensional space, based on their similarity to a given query vector.
Vector search retrieval methods - Vector search retrieval methods are a way of searching for information based on the similarity of vectors in a high-dimensional space.
Keyword indexes - Keyword indexes are a way of organizing and searching for text based on the presence of specific keywords or phrases.
vector database - A vector database is a type of database that stores and retrieves vector data, which can be used for semantic search and other machine learning tasks



In [38]:
from app.together_api_client import TogetherMixtralClient
from pydantic import BaseModel

client = TogetherMixtralClient()

2024-04-09 12:59:03 - Resetting dropped connection: huggingface.co
2024-04-09 12:59:03 - https://huggingface.co:443 "HEAD /mistralai/Mixtral-8x7B-Instruct-v0.1/resolve/main/tokenizer_config.json HTTP/1.1" 200 0


In [48]:
class SubTopicPrompt(BaseModel):
     
    name: str
    description: str
    usage: str | None
    
    @classmethod
    def get_messages(cls, body: str):

        _system_content = """You are a language expert responsible to catagorize sentences into topics. You going to build taxonomy of topics from sentences, and entities you are given.    
            Please ensure that your responses are socially unbiased and positive in nature.
            If a question does not make any sense, or is not factually coherent, explain why instead of answering something not correct. 
            If you don't know the answer, please don't share false information."""

        _user_content_1_examples = """Answers output must confirm to the this JSON format. Insure the JSON is valid. Shorten the answer to make sure the JSON is valid. [/INST] 
            JSON Output: {{
            "topic" :
            {{"name": "leadership", "description": "Leadership the ability of an individual, group, or organization to "lead", influence, or guide other individuals, teams, or entire organizations."}},
            }"""

        _user_content_2_task = """Using the following subtopics that are in the format of "[NAME] - [DESCRIPTION]"
        (for example "policical leader" - "A political leader is someone who is politically active, especially in party politics. Political positions and governments")
        Come up with a topic name and description using the previous JSON output example. 
        The topic name needs to be a common name and descriptive and informative. Here are some examples:  
        1. Topic name "Meeting Management" should be create for the following subtopics:
        - distraction-free virtual meetings - Virtual meetings that are free from distractions and interruptions
        - distraction-free meetings - Meetings that are free from distractions and interruptions, allowing for better engagement and productivity
        - virtual meetings - Meetings held through video conferencing technology
        2. Topic name "Vector Databases" for the following subtopics:
        Vector indexes - Vector indexes are a way of organizing and searching for vectors in a high-dimensional space, based on their similarity to a given query vector.
        Vector search retrieval methods - Vector search retrieval methods are a way of searching for information based on the similarity of vectors in a high-dimensional space.
        vector database - A vector database is a type of database that stores and retrieves vector data, which can be used for semantic search and other machine learning tasks 

        Use the JSON format above to output your answer. Only output valid JSON format. Reduce the length of the answer to make sure the JSON is valid."""

        _user_content_3_subtopics = """Subtopics: {BODY}""".format(BODY=body)

        return [
            {"role": "system", "content": _system_content},
            {"role": "user", "content": _user_content_1_examples},
            {"role": "user", "content": _user_content_2_task},
            {"role": "user", "content": _user_content_3_subtopics}
        ]

In [52]:
answer = await client.generate(body_text = SubTopicPrompt.get_messages(subtopics[3]), model=SubTopicPrompt)   
answer

2024-04-09 13:15:37 - Attempt 1 to generate summary
2024-04-09 13:16:28 - Response status: finished
2024-04-09 13:16:28 - Answer is not None. Stop retrying. Number of attempts 1


SubTopicPrompt(name='Knowledge Representation', description='Knowledge Representation refers to the formalization of information into a machine-readable format, using various structures such as graphs and databases to represent entities and their relationships.', usage={'prompt_tokens': 1243, 'completion_tokens': 108, 'total_tokens': 1351, 'duration': 51509})

In [43]:
answer

SubTopicPrompt(name='meetings', description='Meetings are gatherings where individuals come together to discuss, collaborate, and make decisions. They can be conducted in-person or virtually through video conferencing technology, and can be more productive and engaging when free from distractions and interruptions.', usage={'prompt_tokens': 808, 'completion_tokens': 111, 'total_tokens': 919, 'duration': 1351})